# Relatório — Recomendação de Filmes por Fatoração de Matrizes

Este notebook serve para organizar os resultados do projeto e ajudar a montar o relatório final em formato IEEE.

Ele deve ser executado **depois** do notebook:

```text
notebooks/treinamento.ipynb
```

O notebook de treinamento gera os arquivos dentro da pasta:

```text
results/
```

## 1. Objetivo do projeto

O objetivo deste projeto é comparar três formas de recomendar filmes usando o dataset MovieLens:

1. **Popularidade**;
2. **SVD truncado**;
3. **Matrix Factorization com embeddings**.

A pergunta principal é:

> Um modelo de Matrix Factorization com embeddings consegue recomendar melhor do que SVD simples e popularidade?

A ideia é mostrar como um problema real pode ser modelado usando matrizes, vetores e aproximação de baixa dimensão.

## 2. Modelo matemático usado

A matriz de avaliações é representada por:

\[
R \in \mathbb{R}^{m \times n}
\]

Onde:

- cada linha representa um usuário;
- cada coluna representa um filme;
- cada valor representa a nota dada pelo usuário ao filme.

Como nem todo usuário avalia todo filme, a matriz possui muitos valores ausentes.

### SVD

No SVD, usamos:

\[
R \approx U_k \Sigma_k V_k^T
\]

O valor \(k\) representa a quantidade de fatores latentes usados na aproximação.

### Matrix Factorization

Na Matrix Factorization com embeddings, usamos:

\[
\hat{r}_{ui} = \mu + b_u + b_i + p_u^Tq_i
\]

Onde:

- \(\mu\) é a média geral das notas;
- \(b_u\) é o ajuste do usuário;
- \(b_i\) é o ajuste do filme;
- \(p_u\) é o embedding do usuário;
- \(q_i\) é o embedding do filme.

## 3. Carregar resultados gerados

Esta parte lê os arquivos `.csv` e `.json` criados pelo notebook de treinamento.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np


def find_project_root():
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidate in candidates:
        if (candidate / "results").exists() or (candidate / "data" / "ml_data").exists():
            return candidate
    if Path.cwd().name == "notebooks":
        return Path.cwd().parent
    return Path.cwd()

PROJECT_ROOT = find_project_root()
RESULTS_DIR = PROJECT_ROOT / "results"
IMAGES_DIR = PROJECT_ROOT / "docs" / "images"

print("Raiz do projeto:", PROJECT_ROOT)
print("Pasta de resultados:", RESULTS_DIR)

summary_path = RESULTS_DIR / "summary.json"
metrics_rating_path = RESULTS_DIR / "metrics_rating.csv"
metrics_top10_path = RESULTS_DIR / "metrics_top10.csv"
experiment_k_path = RESULTS_DIR / "experiment_k.csv"
recommendation_examples_path = RESULTS_DIR / "recommendation_examples.csv"

if not summary_path.exists():
    raise FileNotFoundError("Não encontrei results/summary.json. Execute primeiro o notebook treinamento.ipynb")

with open(summary_path, "r", encoding="utf-8") as f:
    summary = json.load(f)

metrics_rating = pd.read_csv(metrics_rating_path)
metrics_top10 = pd.read_csv(metrics_top10_path)
experiment_k = pd.read_csv(experiment_k_path)
recommendation_examples = pd.read_csv(recommendation_examples_path)

print(json.dumps(summary, ensure_ascii=False, indent=2))

## 4. Dados usados

Nesta parte colocamos as principais informações do dataset usado.

In [ ]:
dataset_info = pd.DataFrame([summary["dataset"]])
display(dataset_info.T.rename(columns={0: "valor"}))

## 5. Resultado do Experimento 1 — Erro de previsão

Neste experimento comparamos o erro de previsão das notas.

Métricas usadas:

- **RMSE**: erro quadrático médio com raiz;
- **MAE**: erro absoluto médio.

Quanto menor o valor, melhor o modelo.

In [ ]:
display(metrics_rating)

best_rmse_row = metrics_rating.sort_values("RMSE").iloc[0]
best_mae_row = metrics_rating.sort_values("MAE").iloc[0]

print("Melhor modelo por RMSE:", best_rmse_row["modelo"], "RMSE=", round(best_rmse_row["RMSE"], 4))
print("Melhor modelo por MAE:", best_mae_row["modelo"], "MAE=", round(best_mae_row["MAE"], 4))

### Espaço para imagem

Quando gerar o gráfico, inserir aqui:

```markdown
![Comparação de RMSE](../docs/images/rmse_comparacao.png)
```

## 6. Resultado do Experimento 2 — Recomendações Top-10

Neste experimento analisamos se os filmes recomendados realmente aparecem como relevantes no teste.

Um filme foi considerado relevante quando recebeu nota maior ou igual a 4.

Métricas usadas:

- **Precision@10**: dos 10 recomendados, quantos eram relevantes;
- **Recall@10**: dos relevantes no teste, quantos o modelo conseguiu encontrar.

In [ ]:
display(metrics_top10)

precision_col = [c for c in metrics_top10.columns if c.startswith("Precision@")] [0]
recall_col = [c for c in metrics_top10.columns if c.startswith("Recall@")] [0]

best_precision_row = metrics_top10.sort_values(precision_col, ascending=False).iloc[0]
best_recall_row = metrics_top10.sort_values(recall_col, ascending=False).iloc[0]

print("Melhor modelo por", precision_col + ":", best_precision_row["modelo"], round(best_precision_row[precision_col], 4))
print("Melhor modelo por", recall_col + ":", best_recall_row["modelo"], round(best_recall_row[recall_col], 4))

### Espaço para imagem

Quando gerar o gráfico, inserir aqui:

```markdown
![Comparação Top-10](../docs/images/top10_comparacao.png)
```

## 7. Resultado do Experimento 3 — Efeito do valor de `k`

Aqui comparamos como o número de fatores latentes altera o resultado.

No SVD, `k` é o número de componentes mantidas.

Na Matrix Factorization, `k` é o tamanho dos embeddings de usuários e filmes.

In [ ]:
display(experiment_k)

print("Melhor k do SVD:", summary.get("best_svd_k"))
print("Melhor k da Matrix Factorization:", summary.get("best_mf_k"))

### Espaço para imagem

Quando gerar o gráfico, inserir aqui:

```markdown
![Efeito do valor de k](../docs/images/efeito_k.png)
```

## 8. Resultado do Experimento 4 — Exemplo de recomendação

Aqui mostramos recomendações reais geradas para um usuário do dataset.

In [ ]:
print("Usuário usado no exemplo:", summary.get("chosen_user_id"))
display(recommendation_examples)

### Espaço para imagem

Quando gerar uma tabela/imagem final das recomendações, inserir aqui:

```markdown
![Exemplo de recomendação](../docs/images/exemplo_recomendacao.png)
```

## 9. Interpretação dos resultados

Preencher depois de rodar os experimentos.

### Pergunta 1: Qual modelo teve menor erro?

Resposta inicial:

```text
A preencher com base na tabela de RMSE e MAE.
```

### Pergunta 2: Qual modelo recomendou melhor?

Resposta inicial:

```text
A preencher com base em Precision@10 e Recall@10.
```

### Pergunta 3: A Matrix Factorization superou o SVD?

Resposta inicial:

```text
A preencher comparando os resultados dos dois modelos.
```

### Pergunta 4: O modelo de popularidade foi suficiente?

Resposta inicial:

```text
A preencher. Normalmente a popularidade é simples e serve como baseline, mas não personaliza bem.
```

## 10. Texto-base para o relatório final

Abaixo está um texto inicial que pode ser adaptado para o relatório de 1 página.

In [ ]:
best_rmse_model = metrics_rating.sort_values("RMSE").iloc[0]["modelo"]
best_top10_model = metrics_top10.sort_values(precision_col, ascending=False).iloc[0]["modelo"]

texto = f"""
Este projeto investigou o uso de fatoração de matrizes para recomendação de filmes usando o dataset MovieLens Latest Small.
O problema foi modelado como uma matriz usuário-filme, em que cada entrada representa a nota dada por um usuário a um filme.
Como a maior parte das entradas da matriz é desconhecida, o objetivo foi prever avaliações faltantes e gerar recomendações personalizadas.

Foram comparados três métodos: um baseline de popularidade, SVD truncado e Matrix Factorization com embeddings treinados em PyTorch.
O modelo de popularidade usa a média das avaliações dos filmes, enquanto o SVD aproxima a matriz de avaliações por uma decomposição de baixa dimensão.
A Matrix Factorization representa usuários e filmes por vetores latentes e prevê uma nota pela combinação entre média global, vieses e produto interno dos embeddings.

Nos experimentos de previsão de nota, o melhor modelo por RMSE foi: {best_rmse_model}.
Nas recomendações Top-10, o melhor modelo por {precision_col} foi: {best_top10_model}.
Esses resultados indicam que a representação latente pode capturar padrões de preferência dos usuários melhor do que uma recomendação baseada apenas em popularidade.

As principais limitações são a esparsidade da matriz, a dificuldade com usuários ou filmes novos e o fato de o projeto usar apenas avaliações, sem aproveitar gêneros, tags ou descrições dos filmes.
""".strip()

print(texto)

with open(RESULTS_DIR / "texto_base_relatorio.md", "w", encoding="utf-8") as f:
    f.write(texto)

## 11. Limitações para mencionar na arguição

Pontos importantes para defender:

1. **SVD precisa tratar valores faltantes**: a matriz usuário-filme não está completa, então foi necessário preencher os espaços vazios.
2. **Popularidade não personaliza**: recomenda praticamente os mesmos filmes para todos.
3. **Matrix Factorization depende de histórico**: usuários novos ou filmes novos são difíceis de recomendar.
4. **Só usamos notas**: o modelo não usa gênero, elenco, ano ou tags.
5. **Muitos fatores podem causar overfitting**: aumentar `k` nem sempre melhora o resultado.

Esses pontos ajudam a responder a parte da arguição sobre limitações e decisões do projeto.

## 12. Conclusão esperada

Conclusão em linguagem simples:

> O projeto mostrou que recomendações de filmes podem ser modeladas com Álgebra Linear usando uma matriz usuário-filme. O SVD fornece uma primeira aproximação por decomposição de matriz, enquanto a Matrix Factorization aprende embeddings de usuários e filmes. A comparação com popularidade mostra se o modelo realmente personaliza as recomendações. Os resultados finais indicam qual abordagem apresentou menor erro e melhor qualidade nas recomendações Top-10.